# Round 6 public reproduction: scarcity and lower-need allocation

This notebook is a portable entry point for the public R6 implementation. It loads the
frozen scarcity settings, enumerates the deterministic mechanism cases, and evaluates the
lower-need allocation metrics defined in `src/experiments/scarcity.py`. It does not contain
a second implementation of the model.

The serious held-out evidence and its server provenance are not stored in this repository.
The scheduler-free runner below nevertheless reproduces the public object, development,
and confirmation summaries; it does not submit jobs or access a cluster.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys


def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / 'src').is_dir() and (candidate / 'configs').is_dir():
            return candidate
    raise FileNotFoundError('Open this notebook from inside the repository checkout.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.experiments.scarcity import (
    all_to_lower_allocation,
    build_deterministic_mechanism_cases,
    gaussian_nonpositive_probabilities,
    scarcity_allocation_metrics,
)

settings = json.loads((PROJECT_ROOT / 'configs' / 'scarcity_frozen_settings.json').read_text())
print({'project_root': str(PROJECT_ROOT), 'schema': settings['schema']})

In [ ]:
# Keep this false when opening the notebook. Set true only when you intend to run the
# small public mechanism check. It never launches a cluster workflow.
RUN = False
print({
    'object_episodes': settings['object_screen']['episodes_per_configuration'],
    'development_episodes': settings['development']['episodes_per_environment'],
    'confirmation_episodes': settings['confirmation']['episodes_per_condition'],
    'voi_draws': settings['numerics']['myopic_voi_draws'],
    'support_threshold': settings['thresholds']['support_probability_threshold'],
})

## Portable object, development, and confirmation runner

The shared runner writes `scarcity_oracle_*`, `scarcity_development_*`, and
`scarcity_confirmation_*` summaries under the selected output directory. `smoke` is a
small wiring check. `serious` uses the frozen public episode counts (120 for development
and 1,200 for confirmation) and can be run without a scheduler.

In [ ]:
PORTABLE_MODE = 'smoke'
PORTABLE_OUTPUT_DIR = 'results/round_06_notebook'
portable_command = [
    sys.executable,
    str(PROJECT_ROOT / 'scripts' / 'run_scarcity_public.py'),
    '--mode', PORTABLE_MODE,
    '--output-dir', PORTABLE_OUTPUT_DIR,
]
print(' '.join(portable_command))
if RUN:
    subprocess.run(portable_command, cwd=PROJECT_ROOT, check=True)

SERIOUS_COMMAND = [
    sys.executable,
    str(PROJECT_ROOT / 'scripts' / 'run_scarcity_public.py'),
    '--mode', 'serious',
    '--output-dir', 'results/round_06_serious_public',
]
print('serious/full configuration:', ' '.join(SERIOUS_COMMAND))

In [ ]:
# The Gaussian generator is intentionally unbounded. This reports the theoretical
# probability of a non-positive individual draw rather than silently truncating it.
mu_need = settings['object_screen']['mu_need']
for sigma_need in settings['object_screen']['sigma_need']:
    print(sigma_need, gaussian_nonpositive_probabilities(mu_need, sigma_need))

In [ ]:
if RUN:
    cases = build_deterministic_mechanism_cases()
    rows = []
    for case in cases[:10]:
        config = case['config']
        true_state = case['true_state']
        remaining_time = config.total_time - config.terminate_cost
        allocation = all_to_lower_allocation(
            true_state.need_1,
            true_state.need_2,
            config.learning_per_unit_of_tutoring,
            config.learning_per_unit_of_tutoring - config.delta_learning_per_unit_tutoring,
        )
        metrics = scarcity_allocation_metrics(
            config, true_state, remaining_time, allocation
        )
        rows.append({
            'environment_id': case['environment_id'],
            'lower_effort_identity': metrics['lower_effort_identity'],
            'all_to_lower_match': metrics['all_to_lower_match'],
            'more_to_lower': metrics['more_to_lower'],
        })
    print({'deterministic_case_count': len(cases), 'preview': rows})
else:
    print('RUN=False: definitions loaded; no evaluation executed.')

## Interpreting the public check

`all_to_lower` is an exact allocation comparator. `meet_lower_first` and
`more_to_lower` remain distinct diagnostics. The professor-facing R6 report applies the
prespecified held-out gates and uncertainty intervals to the separate no-search and
active-search classes; this notebook does not recreate those server-side result files.